# Aashish AI — Stage A training (§7.3, P4) — v2

Drives the `training/scripts/` modules, which are tested without a GPU. This notebook is the thin part: it sequences them, checks the environment, and **stops on the licence gate**.

**Why v2 exists.** v1 was correct and still made its own page unresponsive: one cell printed the shard manifest whole — 3,492 file entries, 32,464 lines of JSON — and Jupyter keeps every cell's output in the page and re-sends all of it when the notebook is re-opened. Every output in v2 is therefore **bounded**: a big artifact is summarised (`shard_summary`, `stats_head`), never dumped, whatever its size.

**v2 assumes a fresh session** — empty `/kaggle/working`, the repository arriving as a Dataset archive. It downloads the corpus, builds the text, trains the tokenizer, writes the shards, smoke-tests the loop and starts Stage A. After §1 you never touch the notebook's copy again: the corpus, the artifact and the shards are built inside it, in this session.

What a session that stopped partway needs is §1 of v1's notes: the **cleanup → tokenizer → shard** cells of this notebook (§4–§6) run in order against the already-downloaded text — never a fresh extract of the archive, which would erase the session's own work.

## 0. Setup

In [ ]:
!nvidia-smi

import subprocess
import sys

import torch

print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('python', sys.version.split()[0])
print('bf16 supported:',
      torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)
# A T4 has no bf16 — that is exactly why §7.5 specifies fp16 AMP.
# (Known quirk: is_bf16_supported() reports True on a T4; --amp still uses fp16.)
# No `git rev-parse` here: the archive route carries no .git, and the question
# is answered by the archive's sha256 instead.

In [ ]:
import os
import pathlib
import shutil
import tarfile

WORK = pathlib.Path('/kaggle/working')
MOUNT = pathlib.Path('/kaggle/input')
HOME = WORK / 'aashish-ai-portfolio'


def ensure_repo():
    """Where this session's repository is — creating it if it arrived as an archive.

    Four wrong guesses about this one mount were found the hard way, so this no
    longer guesses. A Kaggle Dataset does **not** mount at `/kaggle/input/<name>`;
    "Your Datasets" land under `/kaggle/input/datasets/<username>/<slug>/`. And
    Kaggle does **not** unpack a `.tgz` for you: the first real session found the
    repository sitting on the mount as a 1.2 MB file, with no `package.json`
    anywhere and the cell dying on `StopIteration`.

    So: two direct paths (a clone, a manual copy), then a search for a
    `package.json` beside `training/notebooks/`, then the archive. All offline,
    and none of it needs an edit the next time the layout changes.
    """
    for candidate in (HOME, WORK / 'newportfolio'):
        if (candidate / 'training').is_dir():
            return candidate
    for marker in MOUNT.rglob('package.json'):
        if (marker.parent / 'training' / 'notebooks').is_dir():
            return marker.parent
    for archive in sorted(list(MOUNT.rglob('*.tgz'))
                          + list(MOUNT.rglob('*.tar.gz'))):
        # `git archive ... HEAD` records members from the repository root, so the
        # members are the repository itself and there is no wrapping directory.
        HOME.mkdir(parents=True, exist_ok=True)
        with tarfile.open(archive) as tar:
            tar.extractall(HOME, filter='data')
        if (HOME / 'training').is_dir():
            return HOME
    return None


repo = ensure_repo()
if repo is None:
    raise SystemExit(
        f'No repository found. Looked in {HOME} and {WORK / "newportfolio"}, and '
        f'for a package.json or an archive under {MOUNT}. Attach the '
        f'aashish-ai-portfolio dataset, or clone the branch into {HOME}.')

# A Dataset mount is read-only, and this run writes data/sources.json (the
# licence verification itself), data/raw/, data/extracted/, data/processed/ and
# the checkpoints long before it trains anything: from the mount, the first
# write dies with OSError: [Errno 30] and the GPU is already allocated. So copy
# it out first, then prove the copy is writable rather than assuming it.
os.chdir(repo)
if pathlib.Path.cwd().is_relative_to(MOUNT):
    if not HOME.exists():
        shutil.copytree(pathlib.Path.cwd(), HOME)
    os.chdir(HOME)
pathlib.Path('.tmp-writable').write_text('ok')
pathlib.Path('.tmp-writable').unlink()

print('cwd:', os.getcwd())
# `python -m` from here puts the repo root on sys.path, which is why every
# command below is run as a module rather than as a path to a file.
!python -c "import tokenizers, numpy, pyarrow; print('tokenizers', tokenizers.__version__, '| numpy', numpy.__version__, '| pyarrow', pyarrow.__version__)"

# ── THE ONE-WAY DOOR — RUN ONCE, THEN NEVER RUN THIS CELL AGAIN ─────────────
# Everything below writes inside the session's own working copy: the downloads,
# the extracted text, the tokenizer artifact and the shards. Re-extracting the
# archive erases all of it — the second session lost a full corpus pass to
# exactly this. If this cell has already run in this session, it is DONE.
for line in sorted(os.listdir('training/notebooks')):
    print(' ', line)

## 2. Licence gate (§7.3)

A source cannot be downloaded until a named person records the licence they read. There is no `--force`, and this is the **first wall, not a formality**: on the first pass it found L3Cube-HingCorpus and DailyDialog are both NonCommercial.

In [ ]:
!python -m training.scripts.fetch_corpus --check

# The gate is already signed (b079775, reviewer "Aashish Kumar", 2026-09-30):
# hindi_wikipedia and simple_english_wikipedia CC-BY-SA-4.0, tinystories
# CDLA-Sharing-1.0, portfolio_instruction own-work. The output above should
# report those sources `ready` and the NonCommercial ones blocked.
#
# Re-verifying by hand looks like:
# !python -m training.scripts.fetch_corpus --verify hindi_wikipedia --spdx CC-BY-SA-4.0 --url https://dumps.wikimedia.org/legal.html --reviewer "Aashish Kumar"
# (It rewrites data/sources.json wholesale — formatting included — so only run
#  it when the record actually changes.)

In [ ]:
# Only sources the gate reports as `ready` can be fetched. Downloads are
# resumable (.part + Range) and hashed, so a restarted session continues.
# tinystories is a different download kind (HF dataset, pinned revision) and is
# deliberately not in this list: Stage A runs on the two Wikipedias.
SOURCES = ['hindi_wikipedia', 'simple_english_wikipedia']

for source in SOURCES:
    !python -m training.scripts.fetch_corpus --source {source}

import glob

for line in sorted(glob.glob('data/raw/*/fetch_manifest.json')):
    print(line)

## 3. Extraction — a download is not text

**This step did not exist in the first notebook, and its absence is why the pipeline could not run.** `fetch_corpus` writes `hiwiki-latest-pages-articles.xml.bz2`; `prepare_data` globs for `*.txt`. Both were tested; nothing connected them.

Text goes to `data/extracted/`, one document per line. That directory is shared on purpose: `prepare_data --raw` reads a *single* directory, so several sources meeting there is how a multi-source corpus is assembled.

In [ ]:
!python -m ai.data.extract --check

for source in SOURCES:
    !python -m ai.data.extract --source {source} --min-chars 200

# For a parquet corpus (TinyStories, Sangraha), inspect the columns rather than
# guessing the text column:
# !python -m ai.data.extract --in data/raw/tinystories --list-columns
# !python -m ai.data.extract --source tinystories --kind parquet --text-column text --min-chars 200
#
# --min-chars is corpus-dependent. 200 suits articles; a short-story or dialogue
# corpus needs less, and extracting 0 documents is a named failure rather than
# an empty file precisely so that this is obvious.

## 4. Corpus pipeline (§7.3) — pass 1: read the stats

Without `--tokenizer`, this pass writes cleanup statistics, the leakage report and `corpus.txt` but **no shards**. It is the cheap pass, and its output is the one worth reading before spending GPU hours.

`stats.json` carries a per-source breakdown (`by_source`) — read it with the `stats_head` cell below rather than `cat` when it is big.

In [ ]:
!python -m training.scripts.prepare_data --raw data/extracted --out data/processed/stage_a

!python -m training.scripts.stats_head data/processed/stage_a/stats.json
!cat data/processed/stage_a/leakage.json

# Read `dropped` by reason. A corpus that keeps 4% of its lines is a filtering
# problem, not a training problem, and no hyperparameter will fix it.
# `corpus.txt` is the train split of the *processed* text — what the tokenizer
# learns from next.
# (leakage.json is a handful of lines; the big file is stats.json, whose
# per-source section is what stats_head exists to bound.)

## 5. Tokenizer (§7.2) — train, then freeze

`--vocab-size 16384` is config A's vocabulary (§7.1), deliberately: the trainer sizes the embedding *from this artifact*, so training at 16,384 makes count_parameters.py's 37,890,560 the number actually trained rather than a design figure. §7.2's window is 12–16k; 16,384 is its top.

In [ ]:
TOKENIZER = 'ai/tokenizer/artifacts/stage-a-16k'

!python -m ai.tokenizer.train --corpus "data/processed/stage_a/corpus.txt" --vocab-size 16384 --min-frequency 2 --out $TOKENIZER

# The train cell already prints the contract checks and the fertility table —
# the separate fertility cell of v1 is gone because it repeated them. The one
# line worth freezing is the artifact's identity, which is what the shard pass
# and every trainer compare against:

In [ ]:
import json

meta = json.load(open(f'{TOKENIZER}/meta.json'))
print('frozen tokenizer_version:', meta['tokenizer_version'], '| vocab', meta['vocab_size'])
print('placeholders reserved:', meta['placeholder_count'])

# tokens/word is the number to judge in the fertility table above: a high Hindi
# figure means the vocabulary is spending its budget on English, and every
# Hindi document then costs more of the block size than an English one.

## 6. Corpus pipeline — pass 2: shard with the frozen tokenizer

**Sharding depends on the tokenizer, so it cannot run before §5.** v1's ancestor sharded with `seed-1k` and then trained with the real tokenizer. Every token id from a 1k vocabulary is still a legal index into a bigger embedding, so nothing failed — the run would have produced a model that encodes text one way and was trained another, and no downstream check could have detected it.

`train_smoke` now compares the shard manifest's `vocab_size` against the tokenizer's and refuses a mismatch. Two passes are safe because the split is a content hash, not a shuffle.

**The manifest is not printed here — and that is the point.** It records every shard file (3,492 of them on the real corpus, ~1.2 MB of JSON); `cat`-ing it is what froze v1's own page. The summary below is the whole decision the manifest serves.

In [ ]:
!python -m training.scripts.prepare_data --raw data/extracted --out data/processed/stage_a --tokenizer $TOKENIZER

!python -m training.scripts.shard_summary data/processed/stage_a/shards

# Check `tokenizer_version` here against the value frozen in §5 — it is the
# field the training guards read, so the question is answered by looking
# rather than by remembering.
# The guard fires even without this check: train_smoke refuses shards whose
# vocab_size differs from the tokenizer's.

## 7. Model + parameter count (§7.1)

At the 16,384-tokenizer this is no longer a design figure: `resolve_config` sizes the embedding from the artifact, so 37,890,560 is the number §8's smoke run will print back.

In [ ]:
!python inference/count_parameters.py --config A --gate

# Two derivations of one number: analytic from the config, then materialised
# from the real module's state_dict, name by name and shape by shape. A mismatch
# exits non-zero — which is the only reason to trust the analytic count.

## 8. Smoke test first (§7.5)

Cheap, and it fails fast on a corpus or tokenizer problem that would otherwise surface 40 minutes into a real run.

In [ ]:
RUN = '/kaggle/working/checkpoints/smoke'

!python -m training.scripts.train_smoke --steps 50 --save-every 10 --amp --shards data/processed/stage_a/shards --tokenizer $TOKENIZER --run-dir $RUN --json /kaggle/working/smoke.json --gate

# ...then prove the resume, in the same session. It must continue the recorded
# stream rather than restart it, and the manifest must report the tokens already
# consumed.
!python -m training.scripts.train_smoke --steps 80 --save-every 10 --amp --shards data/processed/stage_a/shards --tokenizer $TOKENIZER --run-dir $RUN --resume auto --gate


# `--json` records tokens_per_second, but it records it for the *smoke*
# model: 4,769,472 parameters, 4 layers, ctx 256. That is a useful number about
# that model and it is not a number about config A. §9 therefore measures config
# A itself rather than reusing this file.


In [ ]:
import json

smoke = json.load(open('/kaggle/working/smoke.json'))
print('smoke run: config', smoke['config'], '| params', f"{smoke['params']:,}")
print('           batch', smoke['batch'], 'x block', smoke['block'],
      '| tokens/s', f"{smoke['tokens_per_second']:,.0f}")

# Read `config` and `params` before §9 uses this file. They are the two fields
# that decide whether a measurement may be filed under config A, and the first
# version of this notebook filed a 4.8M model's speed under config A's name
# without either of them being looked at.


## 9. Measure config A, then decide the budget (§7.3)

§7.3's ~20 tokens/param is a reference, not a requirement. This turns one
measurement into the number of hours the run actually costs — but only if the
measurement is of **the config being budgeted**.

The first attempt at this cell did not meet that bar. It passed `--from-run
smoke.json` and `estimate_budget` printed

```
config A — 37,890,560 params (37.89M)
  measured           6,270 tokens/s
```

where 6,270 tokens/s belonged to a 4,769,472-parameter, 4-layer, ctx-256 smoke
model. Throughput does not transfer: the small model is bound by kernel-launch
overhead, the large one by arithmetic. Every number below that line inherited
the error, and the step count was then set from it.

So config A is measured directly, at the settings §10 will use, for five steps.
That answers the two questions that matter before nine hours are spent:

* **does it fit** — peak memory, on this card, at this batch;
* **how fast is it** — tokens/sec for *this* model, which `estimate_budget`
  now refuses to accept from any other.


In [ ]:
# 5 steps at §10's exact settings. Costs about a minute. If this says the
# config does not fit, halve --batch and double --grad-accum and probe again —
# the learning update is unchanged, only the memory.
!python -m training.scripts.gpu_probe --config A --tokenizer $TOKENIZER \
    --shards data/processed/stage_a/shards \
    --batch 8 --block 1024 --grad-accum 8 --steps 5 --amp \
    --json /kaggle/working/probe-a.json

# Now the budget is arithmetic on config A's own measured rate.
# `estimate_budget --from-run` refuses a file whose `params` is not this
# config's, so the cell above cannot silently become the cell below's input.
!python -m training.scripts.estimate_budget --config A \
    --from-run /kaggle/working/probe-a.json --hours 9 \
    --dataset-tokens data/processed/stage_a/shards

# Kaggle's free tier gives roughly 30 GPU-hours a week, so size the run to the
# quota. A T4 measures far slower than a desktop GPU: use the printed number,
# not an intuition from a local card.
#
# `micro-steps` is the unit `--steps` counts. One optimizer update happens every
# `--grad-accum` of them, so `--steps 20000` at grad-accum 8 is 2,500 updates.


## 10. Stage A training, resumable

In [ ]:
# Persist checkpoints OUTSIDE the ephemeral session before training (§7.5):
#   → push /kaggle/working/checkpoints/stage-a to a Kaggle Dataset / Drive / HF Hub, or
#   → write to /kaggle/working and Save Version every session
# A session that ends without either loses everything in this directory.
#
# 8 x 1024 x 8 = 65K tokens per learning update, the same as the 16 x 1024 x 4
# this cell ran with the day it died. Activation memory scales with the
# micro-batch and not at all with grad-accum, so the micro-batch is the lever:
# at 16 the first run allocated 12.84 GiB of the T4's 14.56 GiB and raised
# OutOfMemoryError at step 1. §9's probe is what checks this now.
!python -m training.scripts.train_stage_a --config A --tokenizer $TOKENIZER --shards data/processed/stage_a/shards --run-dir /kaggle/working/checkpoints/stage-a --steps 20000 --batch 8 --block 1024 --grad-accum 8 --amp --gate --max-minutes 540 --resume auto

# --max-minutes stops cleanly and writes a checkpoint before the time box ends,
# so an interrupted run is a resume rather than a loss.


In [ ]:
import json

manifest = json.load(open('/kaggle/working/checkpoints/stage-a/RUN_MANIFEST.json'))
print('step', manifest['hyperparameters']['steps'],
      '| tokenizer', manifest['tokenizer_version'],
      '| params', f"{manifest['params']:,}")
print('loss history entries:', len(manifest.get('loss_history', [])))

# Every subsequent session runs the identical training command: it resumes from
# latest.pt and reports the step and the tokens consumed. A resume that silently
# began a new run would look like success here, so check that the step continues
# from where the previous session stopped.

## 11. Validation and samples

In [ ]:
print('<|user|> What projects has Aashish built? <|asst|>')

# Deliberately not implemented yet. Stage A gives a base model that continues
# text; prompts in the §7.4 format are what P5's Stage B teaches it to answer,
# and sampling from the base model with turn tokens it has never learnt is a
# way to conclude the architecture is broken when the data is what is missing.
#
# The two numbers that belong here are the validation loss on the val shards and
# the loss curve. Both come from the training JSON — and a model that has learnt
# synthetic stories and simplewiki will read as fluent and slightly
# encyclopedic. That is a corpus property, not a model defect.

## 12. Export (P6)

In [ ]:
# P6: checkpoint to fp16, then inference/benchmark/benchmark_cpu.py, then the
# parity check, then GGUF/ONNX, then quantize, then measure.
#
# Not scaffolded here on purpose: it needs a checkpoint, and a pipeline built
# against a checkpoint that does not exist yet is a pipeline whose interface is
# invented. Sizes are measured, never estimated in prose.
print('P6 begins once a Stage A checkpoint exists.')